# 6.3 MLOps 与 LLMOps 工具 / MLOps and LLMOps tooling

模型之外的工程环节有哪些，实验跟踪、模型注册、数据版本和线上监控各解决什么问题，MLflow 等工具怎样对应这些环节，以及大模型应用带来了哪些新的要求。

Which engineering stages surround a model, what experiment tracking, model registries, data versioning and production monitoring each solve, how tools such as MLflow map onto these stages, and which new requirements LLM applications bring.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 手写一个实验跟踪器 / An experiment tracker by hand

实验跟踪工具的核心是一种很简单的数据结构。实现 `RunStore`，把每次运行保存为 `root/<run_id>/run.json`。`start_run(params)` 新建一次运行并返回编号，状态为 `running`。`log_metric(run_id, name, value, step)` 追加一条记录，同一指标可以记录多次，形成一条曲线。`end_run(run_id, status='finished')` 更新状态。`get_run(run_id)` 返回 `{"id", "params", "status", "metrics"}`，其中 `metrics[name]` 是按记录顺序排列的 `[step, value]` 列表。`best_run(name, mode)` 在已结束的运行中，按该指标最后一次记录的值返回最好的一次，`mode` 为 `max` 或 `min`，没有记录该指标的运行不参与比较。数据必须写入磁盘，测试会用一个新的 `RunStore` 对象读取同一个目录。

The core of an experiment tracking tool is a very simple data structure. Implement `RunStore`, which saves every run as `root/<run_id>/run.json`. `start_run(params)` creates a run and returns its id, with status `running`. `log_metric(run_id, name, value, step)` appends one record, and a metric can be logged many times to form a curve. `end_run(run_id, status='finished')` updates the status. `get_run(run_id)` returns `{"id", "params", "status", "metrics"}`, where `metrics[name]` is the list of `[step, value]` pairs in the order of logging. `best_run(name, mode)` returns, among the finished runs, the best one by the last logged value of that metric, with `mode` being `max` or `min`, and runs that never logged the metric take no part. The data have to be written to disk. The tests read the same directory with a fresh `RunStore` object.

In [ ]:
import json, pathlib, uuid

class RunStore:
    """A minimal experiment tracker: one directory per run, one JSON file per directory."""
    def __init__(self, root):
        self.root = pathlib.Path(root)
        self.root.mkdir(parents=True, exist_ok=True)

    def start_run(self, params):
        # TODO: create root/<run_id>/run.json with the params, status "running" and no metrics
        raise NotImplementedError

    def log_metric(self, run_id, name, value, step):
        # TODO: append [step, value] to the history of this metric
        raise NotImplementedError

    def end_run(self, run_id, status="finished"):
        raise NotImplementedError

    def get_run(self, run_id):
        raise NotImplementedError

    def best_run(self, name, mode="max"):
        # TODO: among finished runs that logged `name`, compare the last logged values
        raise NotImplementedError

In [ ]:
# 检查 / checks
import tempfile, os
root = tempfile.mkdtemp()
store = RunStore(root)
runs = {}
for lr, final in ((0.1, 0.71), (0.01, 0.84), (0.001, 0.78)):
    rid = store.start_run({"lr": lr, "epochs": 3})
    assert store.get_run(rid)["status"] == "running"
    for step in range(3):
        store.log_metric(rid, "val_acc", final - 0.1 * (2 - step), step)
        store.log_metric(rid, "loss", 1.0 / (step + 1), step)
    store.end_run(rid); runs[lr] = rid
assert len(set(runs.values())) == 3 and all(os.path.isfile(os.path.join(root, r, "run.json")) for r in runs.values())

fresh = RunStore(root)                                   # nothing may live in memory only
r = fresh.get_run(runs[0.01])
assert r["id"] == runs[0.01] and r["params"] == {"lr": 0.01, "epochs": 3} and r["status"] == "finished"
assert [s for s, _ in r["metrics"]["val_acc"]] == [0, 1, 2] and abs(r["metrics"]["val_acc"][-1][1] - 0.84) < 1e-9
assert fresh.best_run("val_acc", "max")["params"]["lr"] == 0.01
assert fresh.best_run("val_acc", "min")["params"]["lr"] == 0.1

crashed = fresh.start_run({"lr": 1.0}); fresh.log_metric(crashed, "val_acc", 0.99, 0)
assert fresh.best_run("val_acc")["params"]["lr"] == 0.01, "a run that is still running is not a candidate"
fresh.end_run(crashed, "failed")
assert fresh.best_run("val_acc")["params"]["lr"] == 0.01, "neither is a failed run"
other = fresh.start_run({"lr": 0.5}); fresh.log_metric(other, "f1", 0.9, 0); fresh.end_run(other)
assert fresh.best_run("val_acc")["params"]["lr"] == 0.01 and fresh.best_run("f1")["params"]["lr"] == 0.5
assert fresh.best_run("unknown") is None
print("ok")

<details><summary><b>提示 1 / Hint 1</b></summary>

写两个内部函数，一个读取某次运行的 JSON，一个把它写回。其余方法都是“读取、修改、写回”。

Write two internal helpers, one that reads the JSON of a run and one that writes it back. Every other method is read, modify, write back.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

运行编号可以用 `uuid.uuid4().hex[:8]`。`best_run` 中遍历 `root` 下的各个目录。

A run id can be `uuid.uuid4().hex[:8]`. In `best_run`, iterate over the directories under `root`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import json, pathlib, uuid

class RunStore:
    """A minimal experiment tracker: one directory per run, one JSON file per directory."""
    def __init__(self, root):
        self.root = pathlib.Path(root)
        self.root.mkdir(parents=True, exist_ok=True)

    def _read(self, run_id):
        return json.loads((self.root / run_id / "run.json").read_text())

    def _write(self, run):
        d = self.root / run["id"]; d.mkdir(exist_ok=True)
        (d / "run.json").write_text(json.dumps(run))

    def start_run(self, params):
        run = {"id": uuid.uuid4().hex[:8], "params": dict(params), "status": "running", "metrics": {}}
        self._write(run)
        return run["id"]

    def log_metric(self, run_id, name, value, step):
        run = self._read(run_id)
        run["metrics"].setdefault(name, []).append([step, value])
        self._write(run)

    def end_run(self, run_id, status="finished"):
        run = self._read(run_id); run["status"] = status
        self._write(run)

    def get_run(self, run_id):
        return self._read(run_id)

    def best_run(self, name, mode="max"):
        runs = [self._read(d.name) for d in self.root.iterdir() if (d / "run.json").exists()]
        runs = [r for r in runs if r["status"] == "finished" and r["metrics"].get(name)]
        if not runs:
            return None
        pick = max if mode == "max" else min
        return pick(runs, key=lambda r: r["metrics"][name][-1][1])
```

</details>

## 2. 用 MLflow 记录同样的实验并对比 / The same experiments with MLflow, compared

用 MLflow 的跟踪接口记录与上一题相同的实验，再把两边读出的结果相互比较。实现 `log_with_mlflow(uri, experiment, configs, curve)`，对 `configs` 中的每组参数开始一次运行，记录参数，并按步记录 `curve(config)` 给出的各步 `val_acc`。实现 `best_with_mlflow(uri, experiment, metric)`，用 `MlflowClient.search_runs` 按指标降序取出最好的一次运行，返回 `(参数字典, 指标值)`。MLflow 把参数保存为字符串，返回前需要转回数值。这个练习需要安装 `mlflow`，数据写入本地的 SQLite 文件，不需要启动服务，也不需要联网。

Record the same experiments as in the previous exercise with the tracking interface of MLflow and compare what the two sides read back. Implement `log_with_mlflow(uri, experiment, configs, curve)`, which starts one run for every set of parameters in `configs`, logs the parameters and logs step by step the `val_acc` values given by `curve(config)`. Implement `best_with_mlflow(uri, experiment, metric)`, which uses `MlflowClient.search_runs` to fetch the best run in descending order of the metric and returns `(parameter dict, metric value)`. MLflow stores parameters as strings, so they have to be converted back to numbers before returning. The exercise needs `mlflow` installed. The data go into a local SQLite file, with no server to start and no network access.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

def log_with_mlflow(uri, experiment, configs, curve):
    """One MLflow run per config: log its params and the val_acc of every step."""
    # TODO: set the tracking uri and the experiment, then start one run per config
    raise NotImplementedError

def best_with_mlflow(uri, experiment, metric="val_acc"):
    """Return (params with numeric values, metric value) of the best run of the experiment."""
    # TODO: search the runs of the experiment ordered by the metric, convert the params back to numbers
    raise NotImplementedError

In [ ]:
# 检查 / checks
import tempfile, os, logging
logging.getLogger("mlflow").setLevel(logging.ERROR); logging.getLogger("alembic").setLevel(logging.ERROR)
configs = [{"lr": 0.1, "epochs": 3}, {"lr": 0.01, "epochs": 3}, {"lr": 0.001, "epochs": 3}]
final = {0.1: 0.71, 0.01: 0.84, 0.001: 0.78}
curve = lambda c: [final[c["lr"]] - 0.1 * (c["epochs"] - 1 - s) for s in range(c["epochs"])]

tmp = tempfile.mkdtemp()
uri = "sqlite:///" + os.path.join(tmp, "mlflow.db")
log_with_mlflow(uri, "lr-sweep", configs, curve)

client = MlflowClient(tracking_uri=uri)
exp = client.get_experiment_by_name("lr-sweep")
runs = client.search_runs([exp.experiment_id])
assert len(runs) == 3 and all(r.info.status == "FINISHED" for r in runs)
history = client.get_metric_history(runs[0].info.run_id, "val_acc")
assert sorted(m.step for m in history) == [0, 1, 2], "one value per step"
assert all(isinstance(v, str) for v in runs[0].data.params.values()), "MLflow stores params as strings"

params, value = best_with_mlflow(uri, "lr-sweep")
assert params == {"lr": 0.01, "epochs": 3} and abs(value - 0.84) < 1e-9, (params, value)

# the hand-written tracker of the previous exercise reaches the same answer from the same runs
store = RunStore(os.path.join(tmp, "runs"))
for c in configs:
    rid = store.start_run(c)
    for s, v in enumerate(curve(c)): store.log_metric(rid, "val_acc", v, s)
    store.end_run(rid)
mine = store.best_run("val_acc")
assert mine["params"] == params and abs(mine["metrics"]["val_acc"][-1][1] - value) < 1e-9
print("ok")

<details><summary><b>提示 1 / Hint 1</b></summary>

记录一侧用 `mlflow.set_tracking_uri(uri)`、`mlflow.set_experiment(experiment)`，然后在 `with mlflow.start_run():` 中调用 `mlflow.log_params(config)` 和 `mlflow.log_metric('val_acc', v, step=i)`。

On the logging side use `mlflow.set_tracking_uri(uri)` and `mlflow.set_experiment(experiment)`, then call `mlflow.log_params(config)` and `mlflow.log_metric('val_acc', v, step=i)` inside `with mlflow.start_run():`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

读取一侧用 `client = MlflowClient(tracking_uri=uri)`，`client.get_experiment_by_name(experiment).experiment_id`，再调用 `client.search_runs([exp_id], order_by=['metrics.val_acc DESC'], max_results=1)`。结果的 `data.params` 和 `data.metrics` 是字典。

On the reading side use `client = MlflowClient(tracking_uri=uri)` and `client.get_experiment_by_name(experiment).experiment_id`, then call `client.search_runs([exp_id], order_by=['metrics.val_acc DESC'], max_results=1)`. `data.params` and `data.metrics` of the result are dicts.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import mlflow
from mlflow.tracking import MlflowClient

def log_with_mlflow(uri, experiment, configs, curve):
    """One MLflow run per config: log its params and the val_acc of every step."""
    mlflow.set_tracking_uri(uri)
    mlflow.set_experiment(experiment)
    for config in configs:
        with mlflow.start_run():
            mlflow.log_params(config)
            for step, value in enumerate(curve(config)):
                mlflow.log_metric("val_acc", value, step=step)

def best_with_mlflow(uri, experiment, metric="val_acc"):
    """Return (params with numeric values, metric value) of the best run of the experiment."""
    client = MlflowClient(tracking_uri=uri)
    exp_id = client.get_experiment_by_name(experiment).experiment_id
    run = client.search_runs([exp_id], order_by=[f"metrics.{metric} DESC"], max_results=1)[0]
    number = lambda s: float(s) if any(ch in s for ch in ".e") else int(s)
    return {k: number(v) for k, v in run.data.params.items()}, run.data.metrics[metric]
```

</details>